<a href="https://colab.research.google.com/github/sandeepa-ukr/AI-Based-Crop-Predection/blob/main/dataexplore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# ML imports
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [5]:
crops = [
    ("Bajra.csv", "bajra"),
    ("Barley.csv", "barley"),
    ("castorseed.csv", "castorseed"),
    ("Cereals.csv", "cereals"),
    ("cotton.csv", "cotton"),
    ("Grains.csv", "grains"),
    ("Gram.csv", "gram"),
    ("Groundnut.csv", "groundnut"),
    ("Jowar.csv", "jowar"),
    ("jute.csv", "jute"),
    ("Lentil.csv", "lentil"),
    ("Linseed.csv", "linseed"),
    ("Maize.csv", "maize"),
    ("mesta.csv", "mesta"),
    ("Moong.csv", "moong"),
    ("Millets.csv", "millets"),
    ("Mustard.csv", "mustard"),
    ("Nigerseed.csv", "nigerseed"),
    ("OilSeeds.csv", "oilseeds"),
    ("Pulses.csv", "pulses"),
    ("ragi.csv", "ragi"),
    ("rice.csv", "rice"),
    ("safflower.csv", "safflower"),
    ("Sesamum.csv", "sesamum"),
    ("Millets.csv", "millets"),
    ("Soybean.csv", "soybean"),
    ("sugarcane.csv", "sugarcane"),
    ("Sunflower.csv", "sunflower"),
    ("tur.csv", "tur"),
    ("Urad.csv", "urad"),
    ("wheat.csv", "wheat"),
]

In [6]:
wether= [
    ("rainfall.csv", "rainfall"),
    ("TEMP_ANNUAL_SEASONAL_MEAN", "temperature")
]

In [7]:
crop_dfs = []
for fname, crop_name in crops:
    tmp = pd.read_csv(fname)
    # tmp = tmp.copy()
    # tmp['Year'] = tmp['Year'].astype(int)
    # tmp['crop_type'] = crop_name
    crop_dfs.append(tmp)

crop_df = pd.concat(crop_dfs, ignore_index=True, axis=0)
print(f"Loaded crop_df shape: {crop_df.shape}")

Loaded crop_df shape: (1045, 39)


In [ ]:
# crop_dfs

In [ ]:
crop_df

In [8]:
print("\n=== Data Quality Checks ===")
print(f"Total rows: {len(crop_df)}")
print(f"Null values per column:")
print(crop_df.isnull().sum())
print(f"\nDataFrame info:")
print(crop_df.info())

# Check unique values in key columns
print(f"\nUnique crops: {crop_df['Item'].nunique() if 'Item' in crop_df.columns else 'Item column not found'}")
print(f"Year range: {crop_df['Year'].min() if 'Year' in crop_df.columns else 'Year column not found'} - {crop_df['Year'].max() if 'Year' in crop_df.columns else 'Year column not found'}")


=== Data Quality Checks ===
Total rows: 1045
Null values per column:
Crop                     5
State                    5
Season                   5
Area_2015_16            22
Area_2016_17             5
Area_2017_18             5
Area_2018_19             5
Area_2019_20             5
Area_2020_21            14
Area_2021_22            24
Area_2022_23             6
Area_2023_24             5
Area_2024_25             5
Area_2025_26           171
Production_2015_16      23
Production_2016_17       5
Production_2017_18       5
Production_2018_19       5
Production_2019_20       5
Production_2020_21      14
Production_2021_22      25
Production_2022_23       7
Production_2023_24       6
Production_2024_25       6
Production_2025_26     172
Yield_2015_16           24
Yield_2016_17            6
Yield_2017_18            6
Yield_2018_19            6
Yield_2019_20            6
Yield_2020_21           14
Yield_2021_22           26
Yield_2022_23            7
Yield_2023_24            6
Yield_2024_2

In [9]:
# Check each file for null values in Crop and State columns
file_null_check = []

for fname, crop_name in crops:
    tmp = pd.read_csv(fname)

    # Count nulls in Crop and State columns
    crop_nulls = tmp['Crop'].isnull().sum() if 'Crop' in tmp.columns else 'Column missing'
    state_nulls = tmp['State'].isnull().sum() if 'State' in tmp.columns else 'Column missing'

    file_null_check.append({
        'File': fname,
        'Crop': crop_name,
        'Rows': len(tmp),
        'Null_Crop': crop_nulls,
        'Null_State': state_nulls,
        'Total_Nulls': tmp.isnull().sum().sum()
    })

# Convert to DataFrame
null_check_df = pd.DataFrame(file_null_check)
print("Files with null values:")
print(null_check_df)

# Show only files that have nulls
print("\nFiles with nulls in Crop or State:")
print(null_check_df[(null_check_df['Null_Crop'] > 0) | (null_check_df['Null_State'] > 0)])

Files with null values:
              File        Crop  Rows  Null_Crop  Null_State  Total_Nulls
0        Bajra.csv       bajra    30          0           0           18
1       Barley.csv      barley    20          0           0            6
2   castorseed.csv  castorseed    14          0           0            0
3      Cereals.csv     cereals    95          0           0          120
4       cotton.csv      cotton    22          0           0            0
5       Grains.csv      grains    96          0           0          108
6         Gram.csv        gram    30          0           0            0
7    Groundnut.csv   groundnut    37          0           0            6
8        Jowar.csv       jowar    32          0           0            0
9         jute.csv        jute     6          0           0            0
10      Lentil.csv      lentil    16          0           0            0
11     Linseed.csv     linseed    14          0           0            0
12       Maize.csv       ma

In [10]:
# Check each file for null values in ALL columns
file_null_check_all = []

for fname, crop_name in crops:
    tmp = pd.read_csv(fname)

    # Get all columns
    all_columns = tmp.columns.tolist()

    # Count nulls for each column
    null_counts = {}
    for col in all_columns:
        null_counts[f'Null_{col}'] = tmp[col].isnull().sum() if col in tmp.columns else 'Column missing'

    # Create record with file info and all null counts
    record = {
        'File': fname,
        'Crop': crop_name,
        'Rows': len(tmp),
        'Columns': len(tmp.columns),
        'Total_Nulls': tmp.isnull().sum().sum()
    }

    # Add null counts for each column
    record.update(null_counts)

    file_null_check_all.append(record)

# Convert to DataFrame
null_check_df_all = pd.DataFrame(file_null_check_all)

# Display basic info
print("=== Null Value Analysis for All Files ===")
print(f"Total files analyzed: {len(null_check_df_all)}")
print("\nFirst few rows (showing only first few columns for brevity):")
print(null_check_df_all.iloc[:, :10].head())

# Find files with ANY null values
files_with_nulls = null_check_df_all[null_check_df_all['Total_Nulls'] > 0]
print(f"\nFiles with ANY null values: {len(files_with_nulls)}")
print(files_with_nulls[['File', 'Rows', 'Total_Nulls']])

# For each file with nulls, show which columns have nulls
print("\n=== Detailed Null Analysis by File ===")
for idx, row in files_with_nulls.iterrows():
    print(f"\n📁 File: {row['File']}")
    print(f"   Total rows: {row['Rows']}")
    print(f"   Total nulls: {row['Total_Nulls']}")

    # Find columns with nulls in this file
    null_columns = []
    for col in row.index:
        if col.startswith('Null_') and row[col] > 0:
            col_name = col.replace('Null_', '')
            null_columns.append(f"{col_name}: {row[col]} nulls")

    if null_columns:
        print("   Columns with nulls:")
        for null_col in null_columns:
            print(f"     - {null_col}")
    else:
        print("   No nulls found (but Total_Nulls > 0 indicates an issue)")

# Summary of which columns have nulls across all files
print("\n=== Summary: Columns with Nulls Across All Files ===")
null_summary = {}
for col in null_check_df_all.columns:
    if col.startswith('Null_'):
        col_name = col.replace('Null_', '')
        total_nulls = null_check_df_all[col].sum()
        if total_nulls > 0:
            null_summary[col_name] = total_nulls

if null_summary:
    null_summary_df = pd.DataFrame(list(null_summary.items()), columns=['Column', 'Total_Nulls_Across_All_Files'])
    null_summary_df = null_summary_df.sort_values('Total_Nulls_Across_All_Files', ascending=False)
    print(null_summary_df)
else:
    print("✅ No null values found in any column across all files!")

=== Null Value Analysis for All Files ===
Total files analyzed: 31

First few rows (showing only first few columns for brevity):
             File        Crop  Rows  Columns  Total_Nulls  Null_Crop  \
0       Bajra.csv       bajra    30       36           18          0   
1      Barley.csv      barley    20       36            6          0   
2  castorseed.csv  castorseed    14       36            0          0   
3     Cereals.csv     cereals    95       36          120          0   
4      cotton.csv      cotton    22       36            0          0   

   Null_State  Null_Season  Null_Area_2015_16  Null_Area_2016_17  
0           0            0                  0                  0  
1           0            0                  0                  0  
2           0            0                  0                  0  
3           0            0                  5                  0  
4           0            0                  0                  0  

Files with ANY null values: 16
    